# 08 — NASA-style RUL Score Audit

This notebook audits the asymmetric validation penalty produced by Notebook 07. It re-computes the score from stored validation predictions, verifies the saved per-row values, compares the aggregation with Notebook 07, and reports which individual cases dominate the score.

**Safety of experiment:** this notebook does not load official test RUL labels, does not retrain models, and does not overwrite a model artifact. Run Notebook 07 first.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / 'reports').exists() and (PROJECT_ROOT.parent / 'reports').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.audit_nasa_score import run_audit

audit = run_audit(PROJECT_ROOT / 'reports')
display(audit['candidate_summary'].round(4))


## How to read the result

- `formula_matches=True` means the saved row-level penalty agrees with the score formula.
- `recomputed_macro_subset_nasa_score_per_case` should match Notebook 07's `macro_subset_nasa_score_per_case`. A material mismatch points to a formula or aggregation discrepancy.
- `top_1_penalty_share_pct` and `top_5_penalty_share_pct` show how concentrated the penalty is in the worst validation cases.
- `pooled_row_sum_nasa_penalty` is a raw sum across validation rows; it is **not directly comparable** to the macro per-case metric.
- `cases_at_exponent_cap` should normally be zero. If it is non-zero, the protective exponent cap of 50 is influencing the score and should be explicitly documented.

The NASA-style penalty is asymmetric: for `d = prediction - truth`, under-prediction uses `exp(-d/13)-1`, while over-prediction uses `exp(d/10)-1`. High values are possible when a small number of cases have large errors.


In [ ]:
# Review the largest penalty-contributing validation cases for cap_350 and uncapped.
top = audit['top_errors']
display(top[top['candidate_id'].isin(['cap_350', 'uncapped'])].head(40).round(4))


In [ ]:
# View how each subset contributes to the score.
display(audit['by_subset'].round(4))
